# Task 8 — Text Classification with Naïve Bayes

### Introduction
Text Classification assigns a predefined category label to a text document.

We build an **SMS Spam Classifier** using a **Naïve Bayes** model trained on TF-IDF features.

### Core Use Cases
- **Spam Filtering**: Automatically routing unsolicited promotional messages to junk.
- **Topic Categorization**: Tagging tickets, news, or reviews into categorized labels (e.g., support, sales).
- **Inappropriate Content Detection**: Censoring toxic inputs or abusive text entries.



### Step 1 — Load and Inspect the Dataset


In [1]:
import pandas as pd
import numpy as np
import urllib.request
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

data_url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv"
data_path = os.path.join("data", "sms_spam.tsv")

if not os.path.exists("data"):
    os.makedirs("data")

if not os.path.exists(data_path):
    print("Downloading SMS Spam Dataset...")
    urllib.request.urlretrieve(data_url, data_path)
    print("Download complete!")

df = pd.read_csv(data_path, sep='\t', names=['label', 'message'])



### Step 2 — Clean, Vectorize, and Train Classifier


In [2]:
import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return ' '.join(text.split())

df['clean_message'] = df['message'].apply(clean_text)

# TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['clean_message'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train NB
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# Evaluate
y_pred = nb_model.predict(X_test)
print("Naïve Bayes Classification Report:")
print(classification_report(y_test, y_pred))



Naïve Bayes Classification Report:
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       966
        spam       1.00      0.77      0.87       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115



### Step 3 — User-Defined Interactive Evaluation Function
Enter custom text messages to evaluate whether they categorize as Ham or Spam.


In [3]:
def evaluate_spam_classifier(message):
    """
    User-defined evaluation function. Cleans the message, extracts TF-IDF, and predicts class.
    """
    clean_msg = clean_text(message)
    feat = vectorizer.transform([clean_msg])
    prediction = nb_model.predict(feat)[0]
    probabilities = nb_model.predict_proba(feat)[0]
    
    spam_prob = probabilities[1] if nb_model.classes_[1] == 'spam' else probabilities[0]
    ham_prob = 1.0 - spam_prob
    
    print(f"Message Input: '{message}'")
    print(f"Classification Verdict: {prediction.upper()}")
    print(f"Probabilities -> Ham: {ham_prob*100:.2f}% | Spam: {spam_prob*100:.2f}%")

# Test custom inputs
evaluate_spam_classifier("Congratulations! You've won a free ticket to the Bahamas. Text WIN to 555 now to claim your cash prize!")
evaluate_spam_classifier("Hey, are we still meeting up for coffee this afternoon? Let me know.")



Message Input: 'Congratulations! You've won a free ticket to the Bahamas. Text WIN to 555 now to claim your cash prize!'
Classification Verdict: SPAM
Probabilities -> Ham: 3.57% | Spam: 96.43%
Message Input: 'Hey, are we still meeting up for coffee this afternoon? Let me know.'
Classification Verdict: HAM
Probabilities -> Ham: 99.86% | Spam: 0.14%
